In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (files are in working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Helper to convert epoch(s) to timezone-aware string
def epoch_to_local_str(epoch):
    return pd.to_datetime(epoch, unit='s', utc=True).tz_convert(tz)

# ---- metric.csv aggregates ----
metric_row_count = len(df_metric)
metric_cmdb_ids = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='unique_cmdb_id')
metric_kpi_counts_df = (
    df_metric.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

metric_min_ts_epoch = int(df_metric['timestamp'].min()) if metric_row_count>0 else None
metric_max_ts_epoch = int(df_metric['timestamp'].max()) if metric_row_count>0 else None
metric_min_ts_local = epoch_to_local_str(metric_min_ts_epoch) if metric_min_ts_epoch is not None else None
metric_max_ts_local = epoch_to_local_str(metric_max_ts_epoch) if metric_max_ts_epoch is not None else None

metrics_overview_df = pd.DataFrame([{
    'total_rows': metric_row_count,
    'unique_cmdb_count': metric_cmdb_ids.shape[0],
    'unique_kpi_count': int(df_metric['kpi_name'].nunique()),
    'min_timestamp_epoch': metric_min_ts_epoch,
    'min_timestamp_local': str(metric_min_ts_local),
    'max_timestamp_epoch': metric_max_ts_epoch,
    'max_timestamp_local': str(metric_max_ts_local)
}])

# ---- trace.csv aggregates ----
trace_row_count = len(df_trace)
trace_cmdb_ids = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='unique_cmdb_id')
trace_name_counts_df = (
    df_trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

trace_min_ts_epoch = int(df_trace['timestamp'].min()) if trace_row_count>0 else None
trace_max_ts_epoch = int(df_trace['timestamp'].max()) if trace_row_count>0 else None
trace_min_ts_local = epoch_to_local_str(trace_min_ts_epoch) if trace_min_ts_epoch is not None else None
trace_max_ts_local = epoch_to_local_str(trace_max_ts_epoch) if trace_max_ts_epoch is not None else None

trace_overview_df = pd.DataFrame([{
    'total_rows': trace_row_count,
    'unique_cmdb_count': trace_cmdb_ids.shape[0],
    'unique_trace_count': int(df_trace['trace_name'].nunique()),
    'min_timestamp_epoch': trace_min_ts_epoch,
    'min_timestamp_local': str(trace_min_ts_local),
    'max_timestamp_epoch': trace_max_ts_epoch,
    'max_timestamp_local': str(trace_max_ts_local)
}])

# Compact outputs: overviews and top counts (limit 50), plus unique cmdb_id lists
metrics_overview_df, metric_kpi_counts_df, metric_cmdb_ids, trace_overview_df, trace_name_counts_df, trace_cmdb_ids

```
Out[1]:
```
```python
# Compose a concise plain-English summary using previously loaded aggregates
mo = metrics_overview_df.iloc[0]
to = trace_overview_df.iloc[0]

top_kpis = metric_kpi_counts_df.head(10).apply(lambda r: f"{r['kpi_name']} ({r['count']})", axis=1).tolist()
top_traces = trace_name_counts_df.head(10).apply(lambda r: f"{r['trace_name']} ({r['count']})", axis=1).tolist()

def sample_list(s, n=12):
    lst = s.tolist()
    if len(lst) <= n:
        return ", ".join(lst)
    return ", ".join(lst[:n]) + f", ... (+{len(lst)-n} more)"

metric_cmdb_sample = sample_list(metric_cmdb_ids)
trace_cmdb_sample = sample_list(trace_cmdb_ids)

summary = (
    "Summary of metric.csv and trace.csv (timestamps shown in UTC+8):\n\n"
    "metric.csv:\n"
    f"- Total rows: {int(mo['total_rows'])}\n"
    f"- Unique cmdb_id count: {int(mo['unique_cmdb_count'])} (examples: {metric_cmdb_sample})\n"
    f"- Unique KPI count: {int(mo['unique_kpi_count'])}\n"
    f"- Time range: epoch {int(mo['min_timestamp_epoch'])} -> {int(mo['max_timestamp_epoch'])}  "
    f"({mo['min_timestamp_local']} -> {mo['max_timestamp_local']})\n"
    "- Top 10 KPI record counts:\n  - " + "\n  - ".join(top_kpis) + "\n\n"
    "trace.csv:\n"
    f"- Total rows: {int(to['total_rows'])}\n"
    f"- Unique cmdb_id count: {int(to['unique_cmdb_count'])} (examples: {trace_cmdb_sample})\n"
    f"- Unique trace count: {int(to['unique_trace_count'])}\n"
    f"- Time range: epoch {int(to['min_timestamp_epoch'])} -> {int(to['max_timestamp_epoch'])}  "
    f"({to['min_timestamp_local']} -> {to['max_timestamp_local']})\n"
    "- Top 10 trace_name record counts:\n  - " + "\n  - ".join(top_traces) + "\n\n"
    "Notes: Top lists show the highest-count KPI/trace names (counts in parentheses). "
    "cmdb_id lists are truncated for brevity."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  unique_cmdb_count  unique_kpi_count  min_timestamp_epoch        min_timestamp_local  max_timestamp_epoch        max_timestamp_local
0       90679                 56               131           1590174000  2020-05-23 03:00:00+08:00           1590177600  2020-05-23 04:00:00+08:00,                     kpi_name  count
0               Disk_io_util   1322
1   Incoming_network_traffic   1319
2                 Send_total   1319
3      Processor_load_15_min   1318
4   Outgoing_network_traffic   1318
..                       ...    ...
45              Sess_Connect    780
46           SctRead_Per_Sec    780
47                  Row_Lock    780
48       DFParaWrite_Per_Sec    780
49                      Hang    780

[50 rows x 2 columns], 0        db_001
1        db_002
2        db_003
3        db_004
4        db_005
        ...    
51    redis_008
52    redis_009
53    redis_010
54    redis_011
55    redis_012
Name: unique_cmdb_id, Length: 56, dtype: object,    total_rows  unique_cmdb_count  unique_trace_count  min_timestamp_epoch        min_timestamp_local  max_timestamp_epoch        max_timestamp_local
0        6240                 10                  76           1590175800  2020-05-23 03:30:00+08:00           1590177540  2020-05-23 03:59:00+08:00,                                          trace_name  count
0                         trace.self.jdbc.row_count    240
1                        trace.self.jdbc.error_rate    240
2                      trace.self.jdbc.duration_p95    240
3                     trace.self.jdbc.duration_mean    240
4                        trace.self.local.row_count    240
..                                              ...    ...
45     trace.to_docker_008.remoteprocess.error_rate     60
46   trace.to_docker_008.remoteprocess.duration_p95     60
47  trace.to_docker_008.remoteprocess.duration_mean     60
48      trace.to_docker_007.remoteprocess.row_count     60
49     trace.to_docker_007.remoteprocess.error_rate     60

[50 rows x 2 columns], 0    docker_001
1    docker_002
2    docker_003
3    docker_004
4    docker_005
5    docker_006
6    docker_007
7    docker_008
8        os_021
9        os_022
Name: unique_cmdb_id, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dfs if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

def epoch_to_local_str(epoch):
    if pd.isna(epoch):
        return None
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# 1) Candidate components intersection
os_list = [f"os_{i:03d}" for i in range(1, 23)]
docker_list = [f"docker_{i:03d}" for i in range(1, 9)]
db_list = [f"db_{i:03d}" for i in range(1, 14)]
possible_components = set(os_list + docker_list + db_list)

metric_cmdb_unique = set(df_metric['cmdb_id'].dropna().unique())
candidate_components = sorted(metric_cmdb_unique.intersection(possible_components))

# Filter metric rows to candidate components for thresholds and windowed analysis
df_metric_candidates = df_metric[df_metric['cmdb_id'].isin(candidate_components)].copy()

# 2) Metric thresholds (global) per (cmdb_id, kpi_name)
g_metric = df_metric_candidates.groupby(['cmdb_id', 'kpi_name'])['value']
metric_p95 = g_metric.quantile(0.95).rename('P95')
metric_p5 = g_metric.quantile(0.05).rename('P5')
metric_thresholds = pd.concat([metric_p95, metric_p5], axis=1).reset_index()

# Failure window
start_ts = 1590175800
end_ts = 1590177600

# Windowed metric rows
df_metric_window = df_metric_candidates[(df_metric_candidates['timestamp'] >= start_ts) &
                                       (df_metric_candidates['timestamp'] <= end_ts)].copy()

# Merge thresholds into window rows for anomaly detection
df_metric_window = df_metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

# Mark anomalies
df_metric_window['is_high'] = df_metric_window['value'] >= df_metric_window['P95']
df_metric_window['is_low'] = df_metric_window['value'] <= df_metric_window['P5']
df_metric_window['is_anomaly'] = df_metric_window['is_high'] | df_metric_window['is_low']

# Aggregate per (cmdb_id, kpi_name) using group apply to access full group
def agg_metric_group(g):
    window_count = int(len(g))
    anomaly_count = int(g['is_anomaly'].sum())
    earliest = int(g.loc[g['is_anomaly'], 'timestamp'].min()) if anomaly_count > 0 else np.nan
    mean_v = float(g['value'].mean()) if window_count>0 else np.nan
    max_v = float(g['value'].max()) if window_count>0 else np.nan
    return pd.Series({
        'window_count': window_count,
        'anomaly_count': anomaly_count,
        'earliest_anom_epoch': earliest,
        'mean_value_in_window': mean_v,
        'max_value_in_window': max_v
    })

if not df_metric_window.empty:
    metric_agg = df_metric_window.groupby(['cmdb_id', 'kpi_name']).apply(agg_metric_group).reset_index()
else:
    # empty frame with expected columns
    metric_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count','anomaly_count','earliest_anom_epoch','mean_value_in_window','max_value_in_window'])

# Merge thresholds so rows without window data still could be examined if needed
metric_agg = metric_agg.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

# Format earliest anomaly ts and tidy columns
metric_agg['earliest_anomaly_ts'] = metric_agg['earliest_anom_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)
metric_agg = metric_agg.drop(columns=['earliest_anom_epoch'])

# Ensure integer counts
metric_agg['anomaly_count'] = metric_agg['anomaly_count'].fillna(0).astype(int)
metric_agg['window_count'] = metric_agg['window_count'].fillna(0).astype(int)

# Select and sort top 20 by anomaly_count desc, then window_count desc
metric_agg_top20 = metric_agg.sort_values(['anomaly_count','window_count'], ascending=[False, False]).head(20)
metric_agg_top20 = metric_agg_top20[['cmdb_id','kpi_name','P95','P5','window_count','anomaly_count','earliest_anomaly_ts','mean_value_in_window','max_value_in_window']]

# 3) Trace thresholds (global) per trace_name
g_trace = df_trace.groupby('trace_name')['value']
trace_p95 = g_trace.quantile(0.95).rename('P95')
trace_p5 = g_trace.quantile(0.05).rename('P5')
trace_thresholds = pd.concat([trace_p95, trace_p5], axis=1).reset_index()

# Windowed trace rows
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
df_trace_window = df_trace_window.merge(trace_thresholds, on='trace_name', how='left', validate='m:1')

# Mark anomalies
df_trace_window['is_high'] = df_trace_window['value'] >= df_trace_window['P95']
df_trace_window['is_low'] = df_trace_window['value'] <= df_trace_window['P5']
df_trace_window['is_anomaly'] = df_trace_window['is_high'] | df_trace_window['is_low']

# Aggregate per trace_name
def agg_trace_group(g):
    window_count = int(len(g))
    anomaly_count = int(g['is_anomaly'].sum())
    earliest = int(g.loc[g['is_anomaly'], 'timestamp'].min()) if anomaly_count>0 else np.nan
    mean_v = float(g['value'].mean()) if window_count>0 else np.nan
    max_v = float(g['value'].max()) if window_count>0 else np.nan
    unique_cmdbs = sorted(g['cmdb_id'].dropna().unique().tolist())
    return pd.Series({
        'window_count': window_count,
        'anomaly_count': anomaly_count,
        'earliest_anom_epoch': earliest,
        'mean_value_in_window': mean_v,
        'max_value_in_window': max_v,
        'unique_cmdbs': unique_cmdbs
    })

if not df_trace_window.empty:
    trace_agg = df_trace_window.groupby('trace_name').apply(agg_trace_group).reset_index()
else:
    trace_agg = pd.DataFrame(columns=['trace_name','window_count','anomaly_count','earliest_anom_epoch','mean_value_in_window','max_value_in_window','unique_cmdbs'])

# Merge thresholds
trace_agg = trace_agg.merge(trace_thresholds, on='trace_name', how='left', validate='1:1')

# Format earliest anomaly ts
trace_agg['earliest_anomaly_ts'] = trace_agg['earliest_anom_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)
trace_agg = trace_agg.drop(columns=['earliest_anom_epoch'])

# Truncate unique cmdbs to at most 8, present as string
def truncate_cmdb_list(lst, n=8):
    if not isinstance(lst, (list, tuple)):
        return ""
    if len(lst) <= n:
        return ", ".join(lst)
    return ", ".join(lst[:n]) + f", ... (+{len(lst)-n} more)"

trace_agg['unique_cmdbs_in_window'] = trace_agg['unique_cmdbs'].apply(lambda x: truncate_cmdb_list(x, 8))
trace_agg = trace_agg.drop(columns=['unique_cmdbs'])

# Ensure integer counts
trace_agg['anomaly_count'] = trace_agg['anomaly_count'].fillna(0).astype(int)
trace_agg['window_count'] = trace_agg['window_count'].fillna(0).astype(int)

# Sort and limit to top 20
trace_agg_top20 = trace_agg.sort_values(['anomaly_count','window_count'], ascending=[False, False]).head(20)
trace_agg_top20 = trace_agg_top20[['trace_name','P95','P5','window_count','anomaly_count','earliest_anomaly_ts','mean_value_in_window','max_value_in_window','unique_cmdbs_in_window']]

# Keep intermediate variables in kernel as requested; final compact outputs (limit 20 rows each)
candidate_components, metric_thresholds.shape, metric_agg_top20, trace_thresholds.shape, trace_agg_top20

```
Out[2]:
```
# Compose a concise plain-English summary using kernel variables (keeps output compact)
n_candidates = len(candidate_components)  # candidate components intersected from metric.csv
n_metric_pairs = metric_thresholds.shape[0]  # number of (cmdb_id, kpi_name) pairs with thresholds
n_trace_names = trace_thresholds.shape[0]  # number of trace_name thresholds

# Summarize top metric anomalies (show up to 5 examples)
m_top = metric_agg_top20.head(5)[['cmdb_id','kpi_name','window_count','anomaly_count','earliest_anomaly_ts']].to_dict('records')
m_lines = []
for r in m_top:
    m_lines.append(f"{r['cmdb_id']} | {r['kpi_name']} — window_count={r['window_count']}, anomaly_count={r['anomaly_count']}, earliest={r['earliest_anomaly_ts']}")

# Summarize top trace anomalies (show up to 5 examples)
t_top = trace_agg_top20.head(5)[['trace_name','window_count','anomaly_count','earliest_anomaly_ts','unique_cmdbs_in_window']].to_dict('records')
t_lines = []
for r in t_top:
    t_lines.append(f"{r['trace_name']} — window_count={r['window_count']}, anomaly_count={r['anomaly_count']}, earliest={r['earliest_anomaly_ts']}, cmdbs=[{r['unique_cmdbs_in_window']}]")

summary = (
    "Summary (compact) of analysis for window 2020-05-23 03:30:00+08 -> 2020-05-23 04:00:00+08:\n\n"
    f"1) Candidate components: {n_candidates} components (intersection of metric cmdb_id and known root-cause lists).\n"
    f"   Examples include db_001..db_013, docker_001..docker_008, os_001..os_022.\n\n"
    f"2) Metrics:\n"
    f"   - (cmdb_id, kpi) pairs with global thresholds computed: {n_metric_pairs}.\n"
    f"   - Top metric anomaly examples (up to 5):\n     - " + "\n     - ".join(m_lines) + "\n\n"
    f"   Interpretation: many top metric pairs show anomaly_count == window_count (i.e., all points in the window were anomalous), with earliest anomalies at 2020-05-23 03:30:00+08.\n\n"
    f"3) Traces:\n"
    f"   - trace names with global thresholds computed: {n_trace_names}.\n"
    f"   - Top trace anomaly examples (up to 5):\n     - " + "\n     - ".join(t_lines) + "\n\n"
    f"   Interpretation: several trace metrics (e.g., trace.self.jdbc.error_rate, trace.self.local.error_rate) had anomalies for every sampled point in the window and involve multiple docker nodes (as shown in cmdb lists).\n\n"
    "Overall conclusion: The failure window (03:30–04:00+08) shows widespread anomalies across multiple DB, docker, and OS components — many KPI and trace series have all or most points in-window flagged as anomalous with earliest occurrences at the window start (2020-05-23 03:30:00+08)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(['db_001', 'db_002', 'db_003', 'db_004', 'db_005', 'db_006', 'db_007', 'db_008', 'db_009', 'db_010', 'db_011', 'db_012', 'db_013', 'docker_001', 'docker_002', 'docker_003', 'docker_004', 'docker_005', 'docker_006', 'docker_007', 'docker_008', 'os_001', 'os_002', 'os_003', 'os_004', 'os_005', 'os_006', 'os_007', 'os_008', 'os_009', 'os_010', 'os_011', 'os_012', 'os_013', 'os_014', 'os_015', 'os_016', 'os_017', 'os_018', 'os_019', 'os_020', 'os_021', 'os_022'], (1779, 4),      cmdb_id             kpi_name         P95          P5  window_count  anomaly_count       earliest_anomaly_ts  mean_value_in_window  max_value_in_window
953   os_007          Disk_rd_ios    0.000000    0.000000            31             31  2020-05-23 03:30:00+0800              0.000000             0.000000
1606  os_020         Disk_io_util    0.000000    0.000000            31             31  2020-05-23 03:30:00+0800              0.794597            24.500000
0     db_001                  ACS    0.020000    0.010000            30             30  2020-05-23 03:30:00+0800              0.015000             0.030000
1     db_001                 AIOS    0.010000    0.000000            30             30  2020-05-23 03:30:00+0800              0.001000             0.010000
10    db_001                 Hang    1.000000    1.000000            30             30  2020-05-23 03:30:00+0800              1.000000             1.000000
15    db_001            MEM_Total  381.902264  381.902264            30             30  2020-05-23 03:30:00+0800            381.902264           381.902264
21    db_001         On_Off_State    1.000000    1.000000            30             30  2020-05-23 03:30:00+0800              1.000000             1.000000
25    db_001        Proc_Used_Pct    1.640000    1.630000            30             30  2020-05-23 03:30:00+0800              1.636000             1.640000
26    db_001   Proc_User_Used_Pct    1.640000    1.630000            30             30  2020-05-23 03:30:00+0800              1.636000             1.640000
28    db_001             Row_Lock    0.000000    0.000000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000
29    db_001      SctRead_Per_Sec    0.000000    0.000000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000
42    db_001  tnsping_result_time   10.000000    0.000000            30             30  2020-05-23 03:30:00+0800              3.666667            10.000000
43    db_002                  ACS    0.020000    0.019500            30             30  2020-05-23 03:30:00+0800              0.019667             0.020000
44    db_002                 AIOS    0.000000    0.000000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000
53    db_002                 Hang    1.000000    1.000000            30             30  2020-05-23 03:30:00+0800              1.000000             1.000000
57    db_002        Login_Per_Sec    0.400000    0.300000            30             30  2020-05-23 03:30:00+0800              0.433333             2.400000
58    db_002            MEM_Total  381.902264  381.902264            30             30  2020-05-23 03:30:00+0800            381.902264           381.902264
64    db_002         On_Off_State    1.000000    1.000000            30             30  2020-05-23 03:30:00+0800              1.000000             1.000000
68    db_002        Proc_Used_Pct    1.340000    1.330000            30             30  2020-05-23 03:30:00+0800              1.333667             1.340000
69    db_002   Proc_User_Used_Pct    1.340000    1.330000            30             30  2020-05-23 03:30:00+0800              1.333667             1.340000, (76, 3),                                         trace_name       P95     P5  window_count  anomaly_count       earliest_anomaly_ts  mean_value_in_window  max_value_in_window                             unique_cmdbs_in_window
38                      trace.self.jdbc.error_rate  0.000000  0.000           240            240  2020-05-23 03:30:00+0800              0.000000             0.000000  docker_001, docker_002, docker_003, docker_004...
42                     trace.self.local.error_rate  0.000000  0.000           240            240  2020-05-23 03:30:00+0800              0.000000             0.000000  docker_001, docker_002, docker_003, docker_004...
37                    trace.self.jdbc.duration_p95  0.006025  0.003           240            228  2020-05-23 03:30:00+0800              0.003512             0.023800  docker_001, docker_002, docker_003, docker_004...
30                       trace.self.csf.error_rate  0.000000  0.000           180            180  2020-05-23 03:30:00+0800              0.001389             0.250000  docker_001, docker_002, docker_003, docker_004...
34                 trace.self.flyremote.error_rate  0.000000  0.000           120            120  2020-05-23 03:30:00+0800              0.000000             0.000000     docker_001, docker_002, docker_003, docker_004
2   trace.from_docker_001.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_007, docker_008
6   trace.from_docker_002.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_007, docker_008
10  trace.from_docker_003.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_005, docker_006
14  trace.from_docker_004.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_005, docker_006
18      trace.from_os_021.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_003, docker_004
22      trace.from_os_022.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_001, docker_002
26                  trace.from_root.osb.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.003472             0.208333                                     os_021, os_022
62    trace.to_docker_005.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_003, docker_004
66    trace.to_docker_006.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_003, docker_004
70    trace.to_docker_007.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_001, docker_002
74    trace.to_docker_008.remoteprocess.error_rate  0.000000  0.000            60             60  2020-05-23 03:30:00+0800              0.000000             0.000000                             docker_001, docker_002
41                   trace.self.local.duration_p95  0.095857  0.008           240             40  2020-05-23 03:30:00+0800              0.042318             0.098000  docker_001, docker_002, docker_003, docker_004...
46    trace.to_docker_001.remoteprocess.error_rate  0.000000  0.000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000                                             os_022
50    trace.to_docker_002.remoteprocess.error_rate  0.000000  0.000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000                                             os_022
54    trace.to_docker_003.remoteprocess.error_rate  0.000000  0.000            30             30  2020-05-23 03:30:00+0800              0.000000             0.000000                                             os_021)```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse variables from kernel: df_metric_window, metric_thresholds, epoch_to_local_str
# If any missing, rebuild minimal required structures (but they should exist per prior steps)
try:
    df_metric_window
except NameError:
    # Recreate df_metric_window minimally if absent (should not happen)
    df_metric_candidates = df_metric[df_metric['cmdb_id'].isin(candidate_components)].copy()
    df_metric_window = df_metric_candidates[(df_metric_candidates['timestamp'] >= 1590175800) & (df_metric_candidates['timestamp'] <= 1590177600)].copy()
    df_metric_window = df_metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    df_metric_window['is_high'] = df_metric_window['value'] >= df_metric_window['P95']
    df_metric_window['is_low'] = df_metric_window['value'] <= df_metric_window['P5']
    df_metric_window['is_anomaly'] = df_metric_window['is_high'] | df_metric_window['is_low']

# helper to compute longest consecutive anomaly streak (consecutive minutes, i.e., ts difference == 60)
def compute_longest_streak(ts_arr, anom_arr):
    if len(ts_arr) == 0:
        return 0, np.nan
    max_len = 0
    max_start = np.nan
    curr_len = 0
    curr_start = np.nan
    prev_ts = None
    for i in range(len(ts_arr)):
        t = int(ts_arr[i])
        is_anom = bool(anom_arr[i])
        if is_anom:
            if curr_len == 0:
                curr_len = 1
                curr_start = t
            else:
                # continue streak only if timestamp is exactly previous + 60
                if prev_ts is not None and t == prev_ts + 60:
                    curr_len += 1
                else:
                    curr_len = 1
                    curr_start = t
            if curr_len > max_len:
                max_len = curr_len
                max_start = curr_start
        else:
            curr_len = 0
            curr_start = np.nan
        prev_ts = t
    return int(max_len), (int(max_start) if not pd.isna(max_start) else np.nan)

# Aggregation per (cmdb_id, kpi_name)
def agg_fault_group(g):
    ts = g['timestamp'].values.astype(int)
    vals = g['value'].values.astype(float)
    anoms = g['is_anomaly'].values.astype(bool)
    window_count = int(len(g))
    anomaly_count = int(anoms.sum())
    earliest_anom_epoch = int(ts[anoms].min()) if anomaly_count>0 else np.nan
    mean_v = float(vals.mean()) if window_count>0 else np.nan
    max_v = float(vals.max()) if window_count>0 else np.nan
    min_v = float(vals.min()) if window_count>0 else np.nan
    longest_streak, streak_start_epoch = compute_longest_streak(ts, anoms)
    return pd.Series({
        'window_count': window_count,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_epoch': earliest_anom_epoch,
        'longest_consecutive_anomaly_streak': longest_streak,
        'streak_start_epoch': streak_start_epoch,
        'mean_value_in_window': mean_v,
        'max_value_in_window': max_v,
        'min_value_in_window': min_v
    })

# Apply aggregation on windowed data grouped by cmdb_id and kpi_name
if not df_metric_window.empty:
    fault_table = df_metric_window.groupby(['cmdb_id','kpi_name']).apply(agg_fault_group).reset_index()
else:
    fault_table = pd.DataFrame(columns=['cmdb_id','kpi_name','window_count','anomaly_count','earliest_anomaly_epoch','longest_consecutive_anomaly_streak','streak_start_epoch','mean_value_in_window','max_value_in_window','min_value_in_window'])

# Merge thresholds (P95,P5) into fault_table (thresholds computed globally earlier)
fault_table = fault_table.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left', validate='1:1')

# Format timestamps columns to UTC+8 strings
fault_table['earliest_anomaly_ts'] = fault_table['earliest_anomaly_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)
fault_table['streak_start_ts'] = fault_table['streak_start_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)

# Compute severity per instructions
def compute_severity(row):
    max_v = row['max_value_in_window']
    min_v = row['min_value_in_window']
    P95 = row['P95']
    P5 = row['P5']
    # severity high-side
    if pd.isna(max_v):
        severity_high = 0.0
    else:
        if P95 > 0:
            severity_high = (max_v - P95) / max(1e-9, max_v)
        else:
            severity_high = max_v
    # severity low-side
    severity_low = 0.0
    if not pd.isna(min_v) and not pd.isna(P5) and min_v < P5:
        severity_low = (P5 - min_v) / max(1e-9, P5)
    return float(max(severity_high, severity_low))

fault_table['severity'] = fault_table.apply(compute_severity, axis=1)

# Prepare numeric fields and handle NaNs
fault_table['window_count'] = fault_table['window_count'].fillna(0).astype(int)
fault_table['anomaly_count'] = fault_table['anomaly_count'].fillna(0).astype(int)
fault_table['longest_consecutive_anomaly_streak'] = fault_table['longest_consecutive_anomaly_streak'].fillna(0).astype(int)

# 4) Filter only pairs with anomaly_count >= max(3, 0.4 * window_count)
def passes_filter(row):
    threshold = max(3, 0.4 * row['window_count'])
    return row['anomaly_count'] >= threshold

fault_table['passes_filter'] = fault_table.apply(passes_filter, axis=1)
fault_table_filtered = fault_table[fault_table['passes_filter']].copy()

# 5) Select and sort top 20 by severity desc then longest_consecutive_anomaly_streak desc
cols_output = [
    'cmdb_id','kpi_name','P95','P5','window_count','anomaly_count',
    'longest_consecutive_anomaly_streak','streak_start_ts','earliest_anomaly_ts',
    'mean_value_in_window','max_value_in_window','min_value_in_window','severity'
]

fault_table_top20 = (fault_table_filtered
                     .sort_values(['severity','longest_consecutive_anomaly_streak'], ascending=[False, False])
                     .head(20)
                     .loc[:, cols_output]
                    )

# Keep full intermediate table in kernel as 'fault_table' (per instruction) and display compact top-20
fault_table_top20

```
Out[3]:
```
```python
# Compose a compact plain-English summary from the fault_table_top20 variable
top = fault_table_top20.head(10).copy()

lines = []
for _, r in top.iterrows():
    lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} — anomalies {r['anomaly_count']}/{r['window_count']}, "
        f"longest_streak={r['longest_consecutive_anomaly_streak']}min (start {r['streak_start_ts']}), "
        f"earliest={r['earliest_anomaly_ts']}, severity={r['severity']:.3f}"
    )

summary = (
    "Fault identification summary for window 2020-05-23 03:30:00+08 -> 2020-05-23 04:00:00+08:\n\n"
    "Top affected series (up to 10):\n- " + "\n- ".join(lines) + "\n\n"
    "Key observations:\n"
    "- The highest-severity series are disk/page-related metrics (Page_pi, Disk_rd_ios, Disk_io_util, Disk_rd_kbs, Page_po) primarily on OS nodes: os_020, os_018, os_019, os_008, os_006.\n"
    "- Many top series have anomaly_count == window_count and longest streak equal to the full window (30–31 minutes), with earliest anomalies at the window start (2020-05-23 03:30:00+08). This indicates sustained/persistent issues rather than sparse noise.\n"
    "- Some DB-level KPI (e.g., db_001 ACS) also appear among top entries but with much lower numeric severity compared to disk metrics.\n\n"
    "Conclusion: The telemetry points toward a sustained disk/page I/O problem starting at the window start on several OS nodes (notably os_020, os_018, os_019). Investigate disk subsystem, paging activity, and affected OS nodes first; also verify related DB metrics for secondary impacts."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name        P95    P5  window_count  anomaly_count  longest_consecutive_anomaly_streak           streak_start_ts       earliest_anomaly_ts  mean_value_in_window  max_value_in_window  min_value_in_window     severity
1628  os_020                   Page_pi   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800             91.386882          2741.606474                  0.0  2741.606474
1607  os_020               Disk_rd_ios   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              9.066667           174.000000                  0.0   174.000000
1606  os_020              Disk_io_util   0.000000  0.00            31             31                                  31  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.794597            24.500000                  0.0    24.500000
1506  os_018              Disk_io_util   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.640104            17.000000                  0.0    17.000000
1556  os_019              Disk_io_util   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.385204            11.556114                  0.0    11.556114
1007  os_008               Disk_wr_ios   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.200000             6.000000                  0.0     6.000000
906   os_006               Disk_wr_ios   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.133333             4.000000                  0.0     4.000000
1057  os_009                Disk_svctm   0.000000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.066667             2.000000                  0.0     2.000000
0     db_001                       ACS   0.020000  0.01            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.015000             0.030000                  0.0     1.000000
432   db_011                       ACS   0.010500  0.01            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.009667             0.020000                  0.0     1.000000
1694  os_021  System_wait_queue_length   2.000000  0.55             6              3                                   1  2020-05-23 03:32:00+0800  2020-05-23 03:32:00+0800              1.166667             2.000000                  0.0     1.000000
1608  os_020               Disk_rd_kbs   0.000003  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              0.088763             1.482370                  0.0     0.999998
1507  os_018               Disk_rd_ios   0.050000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              6.866667           145.000000                  0.0     0.999655
1557  os_019               Disk_rd_ios   0.050000  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              6.100000           136.000000                  0.0     0.999632
1577  os_019                   Page_pi  16.138041  0.00            30             26                                  18  2020-05-23 03:34:00+0800  2020-05-23 03:30:00+0800             47.752966          1393.353602                  0.0     0.988418
67    db_002     Physical_Read_Per_Sec   0.200000  0.00            30             28                                  13  2020-05-23 03:33:00+0800  2020-05-23 03:30:00+0800              0.573333            16.600000                  0.0     0.987952
1558  os_019               Disk_rd_kbs   0.025835  0.00            30             26                                  18  2020-05-23 03:35:00+0800  2020-05-23 03:30:00+0800              0.046732             1.111535                  0.0     0.976757
1026  os_008                   Page_po   4.000000  2.00            30             16                                   6  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              7.133333           126.000000                  2.0     0.968254
924   os_006                   Page_po   4.000000  2.00            30             27                                  18  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800              6.266667           116.000000                  2.0     0.965517
1527  os_018                   Page_pi  72.083765  0.00            30             30                                  30  2020-05-23 03:30:00+0800  2020-05-23 03:30:00+0800             58.834875          1598.973333                  0.0     0.954919```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: fault_table and fault_table_filtered
# If fault_table_filtered isn't present, derive it from fault_table using 'passes_filter' if available
try:
    fault_table
except NameError:
    raise RuntimeError("Required kernel variable 'fault_table' not found. Please run previous fault-detection step.")

if 'fault_table_filtered' not in globals():
    if 'passes_filter' in fault_table.columns:
        fault_table_filtered = fault_table[fault_table['passes_filter']].copy()
    else:
        raise RuntimeError("Cannot find 'fault_table_filtered' or 'passes_filter' column in 'fault_table'.")

# Ensure numeric epoch columns exist
if 'earliest_anomaly_epoch' not in fault_table_filtered.columns:
    # attempt to recover from string timestamps if present
    if 'earliest_anomaly_ts' in fault_table_filtered.columns:
        tz = pytz.timezone('Asia/Shanghai')
        fault_table_filtered['earliest_anomaly_epoch'] = pd.to_datetime(fault_table_filtered['earliest_anomaly_ts']).astype('int64') // 10**9
    else:
        fault_table_filtered['earliest_anomaly_epoch'] = np.nan

# Group by cmdb_id and aggregate required fields
def agg_cmdb(g):
    faulty_kpi_count = int(len(g))
    earliest_epoch = int(g['earliest_anomaly_epoch'].min()) if g['earliest_anomaly_epoch'].notna().any() else np.nan
    latest_epoch = int(g['earliest_anomaly_epoch'].max()) if g['earliest_anomaly_epoch'].notna().any() else np.nan
    max_sev = float(g['severity'].max())
    avg_sev = float(g['severity'].mean())
    total_anom = int(g['anomaly_count'].sum())
    total_window = int(g['window_count'].sum())
    percent_anom = float(total_anom) / float(total_window) if total_window>0 else np.nan
    # top kpis by severity
    topk = g.sort_values('severity', ascending=False)['kpi_name'].dropna().unique().tolist()
    topk_trunc = ", ".join(topk[:8])
    return pd.Series({
        'faulty_kpi_count': faulty_kpi_count,
        'earliest_fault_epoch': earliest_epoch,
        'latest_fault_epoch': latest_epoch,
        'max_severity': max_sev,
        'avg_severity': avg_sev,
        'total_anomalous_points': total_anom,
        'total_window_points': total_window,
        'percent_anomalous': percent_anom,
        'top_kpis': topk_trunc
    })

agg_by_cmdb = fault_table_filtered.groupby('cmdb_id').apply(agg_cmdb).reset_index()

# Convert epoch columns to UTC+8 human-readable strings
tz = pytz.timezone('Asia/Shanghai')
def epoch_to_local_str(epoch):
    if pd.isna(epoch):
        return None
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

agg_by_cmdb['earliest_fault_ts'] = agg_by_cmdb['earliest_fault_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)
agg_by_cmdb['latest_fault_ts'] = agg_by_cmdb['latest_fault_epoch'].apply(lambda x: epoch_to_local_str(x) if not pd.isna(x) else None)

# Select and order columns for output
agg_by_cmdb = agg_by_cmdb[[
    'cmdb_id',
    'faulty_kpi_count',
    'earliest_fault_ts',
    'latest_fault_ts',
    'max_severity',
    'avg_severity',
    'total_anomalous_points',
    'total_window_points',
    'percent_anomalous',
    'top_kpis'
]]

# Sort by faulty_kpi_count desc then max_severity desc, limit top 20
agg_by_cmdb_top20 = agg_by_cmdb.sort_values(['faulty_kpi_count','max_severity'], ascending=[False, False]).head(20).reset_index(drop=True)

# One-line summary counts of faulty cmdb_ids and breakdown by prefix
total_faulty_cmdb = int(agg_by_cmdb.shape[0])
count_os = int(sum(agg_by_cmdb['cmdb_id'].str.startswith('os_')))
count_docker = int(sum(agg_by_cmdb['cmdb_id'].str.startswith('docker_')))
count_db = int(sum(agg_by_cmdb['cmdb_id'].str.startswith('db_')))

summary_line = (
    f"Faulty components: {total_faulty_cmdb} distinct cmdb_id(s) — os: {count_os}, docker: {count_docker}, db: {count_db}."
)

# Final compact outputs
agg_by_cmdb_top20, summary_line

```
Out[4]:
```
Summary (plain English) of aggregated faults by component for the failure window 2020-05-23 03:30:00+08 → 04:00:00+08:

- Overall counts:
  - 43 distinct faulty components found.
  - Breakdown by type: 22 OS nodes (os_*), 8 docker containers (docker_*), 13 DB services (db_*).

- Top affected components (compact highlights, sorted by faulty KPI count / severity):
  1. os_008 — 35 faulty KPIs; earliest fault 2020-05-23 03:30:00+08; latest earliest-KPI time 03:42:00+08; max severity 6.00; total anomalous points 554 of 586 (~94.5%). Top KPIs: Disk_wr_ios, Page_po, Disk_await, CPU_iowait_...
  2. os_006 — 35 faulty KPIs; earliest 03:30:00+08; latest 03:34:00+08; max severity 4.00; total anomalous points 553/586 (~94.4%). Top KPIs: Disk_wr_ios, Page_po, CPU_iowait_time...
  3. os_007 — 34 faulty KPIs; earliest 03:30:00+08; latest 03:48:00+08; max severity ~0.494; anomalous points 533/587 (~90.8%). Top KPIs: Processor_load_1_min, CPU_iowait_time...
  4. os_014 — 30 faulty KPIs; earliest 03:30:00+08; max severity ~0.784; percent anomalous ~93.3%.
  5. os_015 — 30 faulty KPIs; earliest 03:30:00+08; percent anomalous ~93.2%.
  6. db_006 — 29 faulty KPIs; earliest 03:30:00+08; max severity ~0.95; anomalous points 654/717 (~91.2%). Top DB KPIs: AIOS, ACS, Sess_Active...
  7. os_010 — 29 faulty KPIs; max severity ~0.787; percent anomalous ~97.1%.
  8. db_001 — 28 faulty KPIs; max severity 1.00; anomalous points 554/687 (~80.6%).
  9. os_012, os_013, os_001, os_011, os_016, os_009 — each ~26–28 faulty KPIs with high percent anomalous and disk/CPU/page KPIs prominent.
  (Top KPIs lists truncated to keep output compact.)

- Key patterns and observations:
  - Many components show extremely high coverage of anomalies: percent_anomalous is typically >~0.8 and often >0.9, meaning most sampled points in the window were anomalous (not sparse noise).
  - Earliest anomalies for most components begin at the window start (2020-05-23 03:30:00+08), indicating a simultaneous/sustained onset.
  - Highest severities are dominated by disk/page I/O metrics on OS nodes (e.g., os_008 severity 6.0, os_006 severity 4.0). DB services also show many affected KPIs but with generally lower numeric severity compared to top disk metrics.
  - Top KPI types across faulty components: Page_pi/Page_po, Disk_rd_ios, Disk_wr_ios, Disk_io_util, Disk_await, CPU iowait/processor load, and DB KPIs like ACS, AIOS, Physical_Read_Per_Sec.

- Practical next steps (recommended):
  - Prioritize investigation of disk/paging and I/O subsystem on the top OS nodes (os_008, os_006, os_020, os_018, os_019, etc.), since those show highest severity and sustained full-window anomalies.
  - Check storage subsystem health (disks, controllers, SAN/NAS), I/O queues, and OS-level paging activity on those hosts.
  - Correlate with DB KPIs on affected DB nodes (e.g., db_001, db_006, db_011) for secondary impacts and to confirm whether DB symptoms are downstream of OS/disk issues.
  - Review recent configuration/maintenance events and network/storage changes around 03:30+08.

If you want, I can export the top-20 aggregated rows or show the per-component top KPI lists (up to 8 KPIs each) for further drill-down.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  faulty_kpi_count         earliest_fault_ts           latest_fault_ts  max_severity  avg_severity  total_anomalous_points  total_window_points  percent_anomalous                                           top_kpis
0   os_008                35  2020-05-23 03:30:00+0800  2020-05-23 03:42:00+0800      6.000000      0.279502                     554                  586           0.945392  Disk_wr_ios, Page_po, Disk_await, CPU_iowait_t...
1   os_006                35  2020-05-23 03:30:00+0800  2020-05-23 03:34:00+0800      4.000000      0.221698                     553                  586           0.943686  Disk_wr_ios, Page_po, CPU_iowait_time, Process...
2   os_007                34  2020-05-23 03:30:00+0800  2020-05-23 03:48:00+0800      0.493715      0.039768                     533                  587           0.908007  Processor_load_1_min, CPU_iowait_time, Num_of_...
3   os_014                30  2020-05-23 03:30:00+0800  2020-05-23 03:38:00+0800      0.784090      0.036589                     484                  519           0.932563  Processor_load_1_min, Processor_load_5_min, Pr...
4   os_015                30  2020-05-23 03:30:00+0800  2020-05-23 03:38:00+0800      0.630550      0.053818                     478                  513           0.931774  Processor_load_1_min, Disk_wr_ios, Processor_l...
5   db_006                29  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.950000      0.073883                     654                  717           0.912134  AIOS, ACS, Sess_Active, LFSync_Per_Sec, LFPara...
6   os_010                29  2020-05-23 03:30:00+0800  2020-05-23 03:32:00+0800      0.787017      0.044440                     468                  482           0.970954  Disk_avgqu_sz, CPU_iowait_time, Memory_used_pc...
7   db_001                28  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      1.000000      0.088160                     554                  687           0.806405  ACS, Login_Per_Sec, LFSync_Per_Sec, DFParaWrit...
8   os_012                28  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.938737      0.098380                     385                  454           0.848018  Disk_await, Disk_svctm, Disk_io_util, Processo...
9   os_013                28  2020-05-23 03:30:00+0800  2020-05-23 03:34:00+0800      0.584615      0.071998                     416                  455           0.914286  Disk_svctm, Disk_wr_ios, Sent_queue, System_wa...
10  os_001                28  2020-05-23 03:30:00+0800  2020-05-23 03:38:00+0800      0.546382      0.033352                     453                  498           0.909639  Sent_queue, Page_po, Num_of_processes, Agent_p...
11  os_011                28  2020-05-23 03:30:00+0800  2020-05-23 03:32:00+0800      0.407826      0.026576                     389                  450           0.864444  Disk_svctm, Disk_await, Received_packets, Memo...
12  os_016                27  2020-05-23 03:30:00+0800  2020-05-23 03:35:00+0800      0.748981      0.058899                     397                  425           0.934118  Processor_load_1_min, Disk_svctm, Disk_wr_ios,...
13  os_009                26  2020-05-23 03:30:00+0800  2020-05-23 03:35:00+0800      2.000000      0.098179                     404                  444           0.909910  Disk_svctm, Sent_queue, Num_of_processes, Shar...
14  db_011                26  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      1.000000      0.065582                     587                  627           0.936204  ACS, TPS_Per_Sec, Session_pct, SeqRead_Per_Sec...
15  db_005                26  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.672404      0.055604                     570                  627           0.909091  DFParaWrite_Per_Sec, tnsping_result_time, LFSy...
16  db_002                25  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.987952      0.124442                     546                  597           0.914573  Physical_Read_Per_Sec, Login_Per_Sec, tnsping_...
17  db_010                24  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.600000      0.048611                     539                  567           0.950617  tnsping_result_time, ACS, SeqRead_Per_Sec, AIO...
18  db_013                24  2020-05-23 03:30:00+0800  2020-05-23 03:37:00+0800      0.083333      0.004160                     547                  567           0.964727  TPS_Per_Sec, Physical_Read_Per_Sec, ACS, AIOS,...
19  os_002                24  2020-05-23 03:30:00+0800  2020-05-23 03:34:00+0800      0.000435      0.000024                     397                  404           0.982673  Memory_available_pct, Memory_used_pct, Agent_p..., 'Faulty components: 43 distinct cmdb_id(s) — os: 22, docker: 8, db: 13.')```
```